# Phoneme Analysis

**Domain:** Speech & Audio  ·  *from study list*  ·  **runnable:** yes

A refresher on representing and analyzing the *sound units* of speech: what a phoneme is,
how grapheme-to-phoneme (G2P) works, the ARPABET/IPA notations you'll meet in TTS and ASR
pipelines, and the articulatory feature view that powers phonetic distance.

## 1. What & Why

A **phoneme** is the smallest unit of sound that distinguishes one word from another in a
language — swap one and you get a different word (*pat* → *bat*). **Phoneme analysis** is the
practice of converting text or audio into these units and reasoning about them.

It shows up everywhere in speech tech:

- **Text-to-speech (TTS):** front-ends convert graphemes → phonemes before the acoustic model
  (Tacotron 2, VITS, FastSpeech all take phonemes, not raw letters). English spelling is wildly
  irregular (*read*/*read*, *cough*/*through*), so feeding phonemes removes that ambiguity.
- **Automatic speech recognition (ASR):** classic HMM/DNN systems decode to phonemes via a
  pronunciation lexicon; phoneme error rate (PER) is a standard metric.
- **Pronunciation assessment & language learning:** compare a learner's realized phonemes
  against the reference to flag mispronunciations.
- **Search / fuzzy matching:** phonetic hashing (Soundex, Metaphone) and feature-weighted edit
  distance find words that *sound* alike even when spelled differently.

**Reach for it** whenever spelling and sound diverge and you need the sound. **Skip it** when an
end-to-end model already learns the text→audio mapping internally (many modern neural TTS/ASR
systems can be trained on characters directly), or when you only care about the words, not how
they're pronounced.

## 2. Mental Model

Think of speech as a **two-layer alphabet**:

```
  text   "phoneme"        graphemes (letters)   — how it's WRITTEN
            │  G2P (lexicon lookup or model)
            ▼
  sounds  F OW1 N IY0 M    phonemes (ARPABET)    — how it's SAID
            │  each phoneme = a bundle of articulatory features
            ▼
  features  voiceless · labiodental · fricative  — how the mouth MAKES it
```

The key insight: **phonemes are not letters**, and they're not raw audio either — they're a
discrete, finite middle layer (English has ~44). And each phoneme decomposes further into
*articulatory features* (voicing, place, manner of articulation for consonants; height/backness
for vowels). That feature view is what makes "*p* and *b* are nearly the same sound" a
computable fact rather than an intuition: they differ only in **voicing**.

## 3. Key Concepts

- **Phoneme vs. phone vs. allophone.** A *phoneme* is an abstract contrastive unit; a *phone* is
  an actual realized sound; *allophones* are different phones that count as the same phoneme (the
  aspirated *p* in "pin" vs. unaspirated in "spin").
- **Grapheme-to-phoneme (G2P).** Mapping spelling → pronunciation. Done by a **lexicon** (lookup
  dictionary like CMUdict) with a learned model fallback for out-of-vocabulary words.
- **ARPABET.** ASCII phoneme codes for English (`AE`, `T`, `OW1`). Stress is marked with digits
  on vowels: `0` none, `1` primary, `2` secondary. Used by CMUdict and most English TTS.
- **IPA.** The International Phonetic Alphabet — Unicode, language-universal (`/foʊniːm/`). Use it
  for multilingual work; ARPABET is English-only convenience.
- **Articulatory features.** Voicing (voiced/voiceless), place (bilabial, alveolar, velar…), and
  manner (stop, fricative, nasal…) for consonants; height/backness/rounding for vowels.
- **Minimal pair.** Two words differing by exactly one phoneme (*pat*/*bat*) — the evidence that
  two sounds are distinct phonemes in a language.
- **Phonetic distance.** Edit distance between phoneme sequences, optionally weighted by how many
  articulatory features differ — a *b*→*p* substitution costs less than *b*→*s*.

## 4. Setup

No GPU and (for the core examples) no downloads required — pure Python. The optional G2P example
uses NLTK's CMU Pronouncing Dictionary, a small ~3 MB corpus that we gate behind an env var.

```bash
pip install nltk          # optional, only for the CMUdict G2P cell
# Real G2P / IPA toolkits worth knowing (not needed below):
#   pip install g2p_en        # neural English G2P, no lexicon download
#   pip install phonemizer    # multilingual, IPA, wraps eSpeak NG
#   pip install panphon       # IPA -> articulatory feature vectors
```

The two worked examples below are self-contained and run with only the standard library.

In [1]:
import sys
print("Python", sys.version.split()[0])
# Core examples need nothing beyond the stdlib; check NLTK for the optional G2P cell.
try:
    import nltk
    print("nltk", nltk.__version__, "available (optional G2P cell enabled)")
except ImportError:
    print("nltk not installed — optional G2P cell will show call shape only")

Python 3.13.7


nltk 3.9.4 available (optional G2P cell enabled)


## 5. Worked Examples

### Example 1 — Decode ARPABET phonemes into articulatory features

Given a word's pronunciation as ARPABET, expand each phoneme into the features that describe how
the mouth produces it. This is the table that underlies phonetic distance and pronunciation
scoring.

In [2]:
# ARPABET phoneme -> articulatory features (self-contained, no downloads)
ARPABET = {
    # consonants: (kind, voicing, place, manner)
    "P": ("C", "voiceless", "bilabial",    "stop"),
    "B": ("C", "voiced",    "bilabial",    "stop"),
    "T": ("C", "voiceless", "alveolar",    "stop"),
    "D": ("C", "voiced",    "alveolar",    "stop"),
    "K": ("C", "voiceless", "velar",       "stop"),
    "G": ("C", "voiced",    "velar",       "stop"),
    "F": ("C", "voiceless", "labiodental", "fricative"),
    "V": ("C", "voiced",    "labiodental", "fricative"),
    "S": ("C", "voiceless", "alveolar",    "fricative"),
    "Z": ("C", "voiced",    "alveolar",    "fricative"),
    "M": ("C", "voiced",    "bilabial",    "nasal"),
    "N": ("C", "voiced",    "alveolar",    "nasal"),
    # vowels: (kind, voicing, height-backness, manner)
    "AE": ("V", "voiced", "front-low",   "vowel"),
    "IH": ("V", "voiced", "front-high",  "vowel"),
    "AH": ("V", "voiced", "central-mid", "vowel"),
}

def decode(phonemes):
    for ph in phonemes:
        base = ph.rstrip("012")            # strip stress digit on vowels (AE1 -> AE)
        kind, voi, place, manner = ARPABET[base]
        tag = "vowel " if kind == "V" else "conson"
        print(f"  {ph:4} {tag}  {voi:9}  {place:12}  {manner}")

print("cat  ->  K AE1 T")
decode(["K", "AE1", "T"])

cat  ->  K AE1 T
  K    conson  voiceless  velar         stop
  AE1  vowel   voiced     front-low     vowel
  T    conson  voiceless  alveolar      stop


Notice `AE1` carries a stress digit (primary stress) that we strip before lookup. The same
table tells us, mechanically, that `K` and `G` share place (velar) and manner (stop) and differ
only in voicing — which is exactly what the next example exploits.

### Example 2 — Feature-weighted phonetic distance & a minimal-pair detector

A plain Levenshtein distance treats every phoneme substitution as equally costly. Weighting the
substitution by *how many articulatory features differ* gives a far better notion of phonetic
similarity: *p*→*b* (voicing only) is cheap; *p*→*s* (manner + voicing) is dearer.

In [3]:
def feats(ph):
    return ARPABET[ph.rstrip("012")][1:]   # (voicing, place, manner), drop the kind tag

def phone_cost(a, b):
    if a == b:
        return 0.0
    diff = sum(x != y for x, y in zip(feats(a), feats(b)))  # features that differ
    return diff / 3.0                                       # normalize to 0..1

def phonetic_distance(s, t):
    """Levenshtein where substitution cost = articulatory feature distance."""
    n, m = len(s), len(t)
    dp = [[0.0] * (m + 1) for _ in range(n + 1)]
    for i in range(n + 1):
        dp[i][0] = i
    for j in range(m + 1):
        dp[0][j] = j
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            dp[i][j] = min(
                dp[i - 1][j] + 1,                                 # deletion
                dp[i][j - 1] + 1,                                 # insertion
                dp[i - 1][j - 1] + phone_cost(s[i - 1], t[j - 1]) # substitution
            )
    return dp[n][m]

tests = [
    (["P", "AE", "T"], ["B", "AE", "T"]),   # pat / bat  -> voicing only
    (["P", "AE", "T"], ["P", "IH", "T"]),   # pat / pit  -> vowel swap
    (["K", "AE", "T"], ["G", "AE", "T"]),   # cat / gat  -> voicing only
    (["P", "AE", "T"], ["S", "AE", "T"]),   # pat / sat  -> manner + voicing
]
for s, t in tests:
    d = phonetic_distance(s, t)
    label = "minimal pair" if d <= 0.5 else "more distant"
    print(f"{' '.join(s):10} vs {' '.join(t):10}  dist={d:.2f}  ({label})")

P AE T     vs B AE T      dist=0.33  (minimal pair)
P AE T     vs P IH T      dist=0.33  (minimal pair)
K AE T     vs G AE T      dist=0.33  (minimal pair)
P AE T     vs S AE T      dist=0.67  (more distant)


*pat*/*bat* and *cat*/*gat* score 0.33 — a single voicing flip — while *pat*/*sat* scores
higher because the substitution changes both manner and voicing. That ordering is the whole point
of the feature-weighted approach over plain string edit distance.

### Example 3 — Real G2P with the CMU Pronouncing Dictionary (gated)

The examples above hand-built tiny tables. In practice you look pronunciations up in a lexicon.
CMUdict has ~134k English words in ARPABET. The corpus is a small download, so this cell is gated
behind `NLTK_DOWNLOAD=1`; without it, the notebook still runs and shows the call shape.

In [4]:
import os

if os.getenv("NLTK_DOWNLOAD"):
    import nltk
    try:
        nltk.download("cmudict", quiet=True)
        from nltk.corpus import cmudict
        d = cmudict.dict()
        print(f"CMUdict loaded: {len(d):,} words")
        for w in ["phoneme", "tomato", "read"]:
            print(f"  {w:10} -> {d.get(w)}")
    except Exception as e:
        print(f"download/load failed ({type(e).__name__}): run nltk.download('cmudict') manually")
else:
    print("Set NLTK_DOWNLOAD=1 to fetch CMUdict (~3 MB). Expected output:")
    print("  CMUdict loaded: 133,737 words")
    print("  phoneme -> [['F', 'OW1', 'N', 'IY0', 'M']]")
    print("  tomato  -> [['T','AH0','M','EY1','T','OW2'], ['T','AH0','M','AA1','T','OW2']]")
    print("  read    -> [['R','IY1','D'], ['R','EH1','D']]   # homograph: 2 pronunciations")

Set NLTK_DOWNLOAD=1 to fetch CMUdict (~3 MB). Expected output:
  CMUdict loaded: 133,737 words
  phoneme -> [['F', 'OW1', 'N', 'IY0', 'M']]
  tomato  -> [['T','AH0','M','EY1','T','OW2'], ['T','AH0','M','AA1','T','OW2']]
  read    -> [['R','IY1','D'], ['R','EH1','D']]   # homograph: 2 pronunciations


Two things to note in the expected output: *tomato* has two valid pronunciations (the
\"tom-AY-to\"/\"tom-AH-to\" split), and *read* is a **homograph** — same spelling, two
pronunciations depending on tense. A lexicon returns *all* of them; picking the right one needs
part-of-speech or context, which is why G2P front-ends are more than a dictionary lookup.

## 6. Gotchas & Pitfalls

- **Homographs need context.** *read*, *lead*, *tear*, *bass*, *live* have multiple
  pronunciations. A bare lexicon lookup can't choose; you need POS tagging or a sequence model.
- **Out-of-vocabulary words.** Names, slang, and neologisms aren't in CMUdict. Always have a
  learned G2P fallback (e.g. `g2p_en`) or you'll silently drop or mispronounce words.
- **ARPABET is English-only.** Don't try to phonemize French or Mandarin with CMUdict. Use IPA
  via `phonemizer`/eSpeak NG for multilingual work.
- **Stress markers matter.** `AH0` (unstressed schwa) and `AH1` are different in TTS prosody.
  Strip the digits only when you want phoneme identity, not when you care about stress/rhythm.
- **Plain edit distance ≠ phonetic distance.** Treating phonemes as opaque symbols loses the fact
  that *b*/*p* are near-twins. Weight by articulatory features (or use `panphon`) for anything
  perceptual.
- **Phoneme set mismatch.** CMUdict uses 39 phones; IPA is open-ended; eSpeak has its own scheme.
  Mixing inventories across tools without a mapping table produces garbage.
- **Silent normalization bugs.** Numbers, abbreviations, and punctuation ("Dr.", "1990s", "$5")
  must be expanded to words *before* G2P. This text-normalization step is where most TTS
  front-end errors actually live.

## 7. When to Use vs Alternatives

| Approach | Best for | Trade-off |
|---|---|---|
| **Lexicon lookup (CMUdict)** | Fast, exact, English, in-vocabulary words | No OOV handling, English-only, homograph-blind |
| **Neural G2P (`g2p_en`, seq2seq)** | OOV words, names, end-to-end English G2P | Heavier, occasional errors, still mostly English |
| **`phonemizer` / eSpeak NG** | Multilingual, IPA output, TTS front-ends | External binary dependency; quality varies by language |
| **`panphon`** | IPA → articulatory feature vectors, phonetic distance | IPA input only; it's analysis, not G2P |
| **Soundex / Metaphone** | Cheap fuzzy name matching, search | Coarse, lossy; not a real pronunciation |
| **End-to-end neural TTS/ASR on characters** | When you can train on lots of data | Learns G2P implicitly; less control, harder to debug pronunciation |

**Rules of thumb.** English TTS front-end → CMUdict lexicon + neural fallback for OOV.
Multilingual → `phonemizer`/IPA. Phonetic similarity/distance research → `panphon` feature
vectors. Quick name-matching in a database → Metaphone. Only go fully end-to-end on characters
when you have the data and don't need to inspect or correct pronunciations.

## 8. Resources

- **CMU Pronouncing Dictionary** — the canonical English ARPABET lexicon:
  http://www.speech.cs.cmu.edu/cgi-bin/cmudict
- **ARPABET reference (Wikipedia)** — the phoneme codes and stress digits at a glance:
  https://en.wikipedia.org/wiki/ARPABET
- **`phonemizer`** — multilingual text→IPA, the de-facto TTS front-end tool:
  https://github.com/bootphon/phonemizer
- **`g2p_en`** — lightweight neural English G2P with OOV handling:
  https://github.com/Kyubyong/g2p
- **`panphon`** — IPA segments to articulatory feature vectors + phonetic distance:
  https://github.com/dmort27/panphon
- **International Phonetic Alphabet (IPA) chart** — the universal sound inventory:
  https://www.internationalphoneticassociation.org/content/ipa-chart